# LLM Playground on Isambard

Run **Nemotron 3 Super (120B-A12B)** across the four GH200s of one Isambard node.

Set up first (see `README.md`): `bash setup_environment.sh` on a login node,
`sbatch tunnel.sbatch`, then connect and pick the `.venv` kernel.

## 1. Environment check

In [1]:
import os

# LD_PRELOAD pointing at the system NCCL breaks `import torch`. It cannot be
# fixed here -- the loader applies it at exec time -- so detect and stop.
# tunnel.sbatch unsets it for you. (README, trap 2.)
if os.environ.get("LD_PRELOAD"):
    raise SystemExit(
        f"LD_PRELOAD is set:\n    {os.environ['LD_PRELOAD']}\n\n"
        "Run `unset LD_PRELOAD` in the shell, then restart the kernel."
    )

# $HOME is capped at 100 GiB and the checkpoint is ~230 GiB, so the cache has
# to live elsewhere. No need to look up your project code: the BriCS modules
# export $SCRATCH (5 TiB, per-user) and $PROJECTDIR (shared), and project dirs
# are group-owned brics.<code> so it is derivable from `id` either way.
if not os.environ.get("HF_HOME"):
    import subprocess
    for var in ("SCRATCH", "SCRATCHDIR", "PROJECTDIR"):
        p = os.environ.get(var)
        if p and os.access(p, os.W_OK):
            os.environ["HF_HOME"] = os.path.join(p, "hf"); break
    else:
        groups = subprocess.run(["id", "-Gn"], capture_output=True, text=True).stdout.split()
        for g in groups:
            if g.startswith("brics."):
                p = "/projects/" + g.split(".", 1)[1]
                if os.access(p, os.W_OK):
                    os.environ["HF_HOME"] = os.path.join(p, "hf"); break

print(f"HF_HOME = {os.environ.get('HF_HOME', '~/.cache/huggingface -- too small, see README')}")

import torch, transformers

print(f"torch          {torch.__version__}")
print(f"transformers   {transformers.__version__}")
print(f"CUDA available {torch.cuda.is_available()}")
print(f"GPUs visible   {torch.cuda.device_count()}")

# On ARM, plain `pip install torch` gives a CPU-only or CUDA-13 wheel, both of
# which fail silently or late. pyproject.toml pins cu126. (README, trap 1.)
assert "+cu" in torch.__version__, f"{torch.__version__}: CPU-only wheel, see pyproject.toml"
assert torch.cuda.is_available(), "No GPU -- login node, or job lacks --gpus-per-node=4?"

total = sum(torch.cuda.get_device_properties(i).total_memory
            for i in range(torch.cuda.device_count()))
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU[{i}] {p.name}  sm_{p.major}{p.minor}  {p.total_memory/2**30:.1f} GiB")
print(f"\naggregate GPU memory: {total/2**30:.1f} GiB")

HF_HOME = /projects/a5k/public/hf
torch          2.13.0+cu126
transformers   5.14.1
CUDA available True
GPUs visible   4
  GPU[0] NVIDIA GH200 120GB  sm_90  95.0 GiB
  GPU[1] NVIDIA GH200 120GB  sm_90  95.0 GiB
  GPU[2] NVIDIA GH200 120GB  sm_90  95.0 GiB
  GPU[3] NVIDIA GH200 120GB  sm_90  95.0 GiB

aggregate GPU memory: 380.0 GiB


## 2. Load the model

~225 GiB of weights, so it needs 4 GPUs (380 GiB aggregate). `device_map="auto"`
shards it: one process, layers spread across GPUs, activations copied between
them. No NCCL, no process group — which is why no NCCL module is loaded.

Takes ~110 s from Lustre. The "fast path is not available" warning is expected:
the Mamba2 CUDA kernels have no ARM wheels, so `transformers` uses a correct
pure-PyTorch fallback.

In [2]:
import time
from transformers import pipeline

MODEL = "nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-BF16"

t0 = time.time()
pipe = pipeline(
    "text-generation",
    model=MODEL,
    dtype=torch.bfloat16,   # note: `dtype`, not the older `torch_dtype`
    device_map="auto",      # shard across all visible GPUs
)
print(f"loaded in {time.time()-t0:.1f}s")

[transformers] The fast path is not available because one of `(selective_state_update, causal_conv1d_fn, causal_conv1d_update)` is None. Falling back to the naive implementation. To install follow https://github.com/state-spaces/mamba/#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/763 [00:00<?, ?it/s]

loaded in 118.4s


In [3]:
from collections import Counter

# Which GPU did each layer land on?
spread = Counter(pipe.model.hf_device_map.values())
print("layers per GPU:", dict(sorted(spread.items())))

for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f"  GPU[{i}] {(total-free)/2**30:5.1f} GiB used / {total/2**30:.1f} GiB")

layers per GPU: {0: 24, 1: 25, 2: 24, 3: 18}
  GPU[0]  62.8 GiB used / 95.0 GiB
  GPU[1]  67.2 GiB used / 95.0 GiB
  GPU[2]  67.2 GiB used / 95.0 GiB
  GPU[3]  52.1 GiB used / 95.0 GiB


## 3. Generating text

This is a reasoning model and reasoning is **on by default** — a short prompt
will spend its whole budget inside a `<think>` block and never answer. Turn it
off with `enable_thinking=False`, which is a *chat template* argument, not a
`generate` argument: passing it to the pipeline raises `ValueError`. So render
the prompt first, then generate from the string.

Two harmless warnings appear on every call (a stale `max_length=20` in the
model's `generation_config`, and a `transformers` deprecation notice).

In [ ]:
def chat(user_message, max_new_tokens=1024, think=True, **kw):
    """Render the chat template, then generate. Returns the assistant's reply."""
    prompt = pipe.tokenizer.apply_chat_template(
        [{"role": "user", "content": user_message}],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=think,      # a CHAT TEMPLATE kwarg, not a generate kwarg
    )
    out = pipe(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        return_full_text=False,             # reply only, not the prompt back
        clean_up_tokenization_spaces=False, # destructive for BPE tokenizers
        **kw,
    )
    return out[0]["generated_text"].strip()

In [6]:
t0 = time.time()
reply = chat("Explain what a supercomputer interconnect does, in two sentences.")
print("\n\n" + reply)

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)




We need to answer: explain what a supercomputer interconnect does, in two sentences. Should be concise. Provide two sentences. No extra fluff.

</think>

A supercomputer interconnect is the high‑speed network that links the thousands of compute nodes, memory modules, and I/O systems, enabling them to exchange data and synchronize operations with minimal latency and maximal bandwidth. It determines how efficiently the system can scale and perform parallel workloads by providing the communication backbone for messages, shared memory coherence, and collective operations.


In [ ]:
t0 = time.time()
reply = chat("Tell me about the IsambardAI compute cluster.", think=True, max_new_tokens=8192)
print("\n\n" + reply)

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)




Actually, there is no widely recognized or publicly documented compute cluster named **“IsambardAI”** as of now (June 2024).

It’s possible you may be referring to one of the following:

### 1. **Isambard (the UK’s ARM-based supercomputer)**
The name *Isambard* is most famously associated with **Isambard**, a UK national supercomputing resource hosted at the **Met Office** and operated by the **UK Met Office** in collaboration with the **UKRI** (UK Research and Innovation).  
- **Isambard** (launched 2018) was one of the world’s first large-scale ARM-based supercomputers, built using **Cray XC50** architecture with **ARMv8-A** processors (specifically, Cavium ThunderX2 CPUs).  
- It was designed for energy-efficient high-performance computing (HPC), particularly for weather and climate modeling.  
- A successor, **Isambard 2**, was deployed in 2021, featuring **Arm-based Neoverse N1** processors (from Fujitsu’s A64FX, similar to those in Fugaku) and GPUs for AI workloads.  
- Isambar

---

**Next:** jobs are capped at 24 h — save to project storage, not the node.
Models too large for one node (the 550B Ultra, ~1.1 TB) need multi-node serving
with vLLM, where the Slingshot and NCCL setup skipped here starts to matter.